# 01 — Experiment & Metrics

This notebook covers **Gap 1**: a single, standard evaluation format that works identically everywhere.

Topics covered:
- Creating an `Experiment` with all config options
- Logging predictions from lists, numpy arrays, and in batches
- All five metrics explained: Accuracy, AUC-ROC, F1, Log-loss, Brier score
- `result.summary()` — the enhanced table
- `result.report()` — the HTML report
- Cohort-level breakdowns (per region, per segment)
- `stop_early` — deciding a winner before `min_samples` is reached
- All three methods: `bayesian`, `sequential`, `bandit`
- `save()` and `load()` — persisting experiment state
- What `result.ready`, `result.confidence`, `result.winner` mean

In [1]:
# !pip install evalbridge scikit-learn -q

## 1. Build two models to compare

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = make_classification(
    n_samples=3000, n_features=20, n_informative=12,
    n_redundant=4, random_state=0
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

baseline   = LogisticRegression(random_state=0, max_iter=1000).fit(X_train, y_train)
challenger = GradientBoostingClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)

base_preds = baseline.predict_proba(X_test)[:, 1]
chal_preds = challenger.predict_proba(X_test)[:, 1]

print(f"Baseline   AUC: {roc_auc_score(y_test, base_preds):.4f}")
print(f"Challenger AUC: {roc_auc_score(y_test, chal_preds):.4f}")
print(f"Test set size : {len(y_test)} samples")

Baseline   AUC: 0.7896
Challenger AUC: 0.9592
Test set size : 1200 samples


## 2. The simplest possible experiment (10 lines)

In [3]:
from evalbridge import Experiment

exp = Experiment("churn_v2_vs_v3")
exp.log("baseline",   y_true=y_test.tolist(), y_pred=base_preds.tolist())
exp.log("challenger", y_true=y_test.tolist(), y_pred=chal_preds.tolist())

result = exp.evaluate()
result.summary()


┌──────────────┬──────────┬──────────┬────────┬──────────┬─────────────┬────────────┬───┐
│ Model        │ Accuracy │ AUC-ROC  │   F1   │ Log-loss │ Brier score │  Samples   │   │
├──────────────┼──────────┼──────────┼────────┼──────────┼─────────────┼────────────┼───┤
│ baseline     │    0.707 │    0.790 │  0.709 │    0.553 │       0.187 │      1,200 │    │
│ challenger   │    0.892 │    0.959 │  0.892 │    0.287 │       0.083 │      1,200 │ ◀ │
└──────────────┴──────────┴──────────┴────────┴──────────┴─────────────┴────────────┴───┘
  Confidence  [████████████████████] 100.0%
  Status      READY ✓
  Winner      challenger



## 3. Reading the result object

In [4]:
print(f"winner      : {result.winner}")
print(f"confidence  : {result.confidence:.4f}   (how certain we are)")
print(f"ready       : {result.ready}         (min_samples + min_confidence both met)")
print()
print("baseline metrics:")
for k, v in result.metrics['baseline'].items():
    print(f"  {k:<14} {v}")
print()
print("challenger metrics:")
for k, v in result.metrics['challenger'].items():
    print(f"  {k:<14} {v}")

winner      : challenger
confidence  : 1.0000   (how certain we are)
ready       : True         (min_samples + min_confidence both met)

baseline metrics:
  accuracy       0.7066666666666667
  auc_roc        0.7895880272592658
  f1             0.7086092715231788
  log_loss       0.552924661240668
  brier_score    0.18734293949557318
  n_samples      1200

challenger metrics:
  accuracy       0.8925
  auc_roc        0.9591666643515367
  f1             0.8918692372170998
  log_loss       0.2873886590902642
  brier_score    0.08301202219766526
  n_samples      1200


## 4. All five metrics explained

| Metric | Range | Better when | What it measures |
|---|---|---|---|
| `accuracy` | 0 – 1 | Higher | Fraction of correct predictions after thresholding at 0.5 |
| `auc_roc` | 0 – 1 | Higher | Probability a random positive scores above a random negative. Threshold-independent |
| `f1` | 0 – 1 | Higher | Harmonic mean of precision and recall |
| `log_loss` | 0 – ∞ | Lower | Penalises confident wrong predictions. Measures calibration |
| `brier_score` | 0 – 1 | Lower | Mean squared error between probabilities and labels |

Use **AUC-ROC** as the primary ranking metric. Use **Brier score** when you care about calibration (probabilities, not just rankings).

In [5]:
# Compare each metric between the two models
b = result.metrics['baseline']
c = result.metrics['challenger']

print(f"{'Metric':<14} {'Baseline':>10} {'Challenger':>12} {'Delta':>10} {'Better'}")
print("-" * 58)
for key in ['accuracy', 'auc_roc', 'f1', 'log_loss', 'brier_score']:
    bv, cv = b[key], c[key]
    delta = cv - bv
    higher_better = key not in ('log_loss', 'brier_score')
    challenger_better = (delta > 0) if higher_better else (delta < 0)
    winner_label = "challenger ✓" if challenger_better else "baseline"
    print(f"{key:<14} {bv:>10.4f} {cv:>12.4f} {delta:>+10.4f}  {winner_label}")

Metric           Baseline   Challenger      Delta Better
----------------------------------------------------------
accuracy           0.7067       0.8925    +0.1858  challenger ✓
auc_roc            0.7896       0.9592    +0.1696  challenger ✓
f1                 0.7086       0.8919    +0.1833  challenger ✓
log_loss           0.5529       0.2874    -0.2655  challenger ✓
brier_score        0.1873       0.0830    -0.1043  challenger ✓


## 5. Full Experiment config — every parameter explained

In [6]:
exp_full = Experiment(
    name="churn_full_config",

    # Which statistical test to use:
    # "bayesian"    — Beta-Binomial model, P(challenger > baseline) via Monte Carlo
    # "sequential"  — SPRT, can stop early without inflating false positive rate
    # "bandit"      — same as bayesian but meant to be used with as_bandit()
    method="bayesian",

    # Minimum confidence before result.winner is set (not "inconclusive")
    min_confidence=0.95,

    # Minimum samples per model before result.ready is True
    min_samples=200,

    # If True: declare winner before min_samples IF confidence is already >= min_confidence
    stop_early=True,

    # Cohort dimensions to track per-segment results
    cohorts={"region": ["EU", "US", "APAC"]},

    # Auto-initialize DriftDetector when go_live() is called
    alert_on_drift=True,

    # PSI threshold above which drift.alert fires
    drift_threshold=0.2,
)

print(f"Experiment '{exp_full.name}' created")
print(f"  method        : {exp_full.method}")
print(f"  min_confidence: {exp_full.min_confidence}")
print(f"  min_samples   : {exp_full.min_samples}")
print(f"  stop_early    : {exp_full.stop_early}")
print(f"  cohorts       : {exp_full.cohorts}")

Experiment 'churn_full_config' created
  method        : bayesian
  min_confidence: 0.95
  min_samples   : 200
  stop_early    : True
  cohorts       : {'region': ['EU', 'US', 'APAC']}


## 6. Logging: lists, numpy arrays, and batches

In [7]:
exp_log = Experiment("log_demo", min_samples=5)

# Log from Python lists
exp_log.log("baseline", y_true=[1, 0, 1], y_pred=[0.8, 0.2, 0.7])

# Log from numpy arrays — works identically
exp_log.log("baseline", y_true=np.array([0, 1, 0]), y_pred=np.array([0.3, 0.9, 0.1]))

# Log incrementally — call log() many times to accumulate
for i in range(4):
    exp_log.log("challenger", y_true=[1, 0], y_pred=[0.85, 0.15])

# Log with cohort tag
exp_log.log("baseline",   y_true=[1, 0, 1], y_pred=[0.8, 0.2, 0.7], cohort={"region": "EU"})
exp_log.log("challenger", y_true=[1, 0, 1], y_pred=[0.9, 0.1, 0.8], cohort={"region": "EU"})

print(f"baseline samples  : {len(exp_log._data['baseline']['y_true'])}")
print(f"challenger samples: {len(exp_log._data['challenger']['y_true'])}")
print(f"cohort data keys  : {list(exp_log._cohort_data.keys())}")

baseline samples  : 9
challenger samples: 11
cohort data keys  : ['region']


## 7. Cohort-level breakdowns

In [8]:
rng = np.random.default_rng(1)

exp_cohort = Experiment("cohort_demo", min_samples=50, min_confidence=0.7)

regions = {"EU": 300, "US": 300, "APAC": 150}

for region, n in regions.items():
    yt = rng.integers(0, 2, n).tolist()

    # EU: challenger much better; US: similar; APAC: baseline slightly better
    if region == "EU":
        bp = [min(1, max(0, y + rng.normal(0.05, 0.3))) for y in yt]
        cp = [min(1, max(0, y + rng.normal(0.35, 0.15))) for y in yt]
    elif region == "US":
        bp = [min(1, max(0, y + rng.normal(0.2, 0.25))) for y in yt]
        cp = [min(1, max(0, y + rng.normal(0.22, 0.25))) for y in yt]
    else:  # APAC
        bp = [min(1, max(0, y + rng.normal(0.3, 0.2))) for y in yt]
        cp = [min(1, max(0, y + rng.normal(0.1, 0.35))) for y in yt]

    exp_cohort.log("baseline",   y_true=yt, y_pred=bp, cohort={"region": region})
    exp_cohort.log("challenger", y_true=yt, y_pred=cp, cohort={"region": region})

result_cohort = exp_cohort.evaluate()

print(f"Global winner: {result_cohort.winner} (confidence: {result_cohort.confidence:.3f})")
print()
print(f"{'Region':<8} {'Winner':<12} {'Confidence':>12} {'Delta Acc':>12}")
print("-" * 48)
for region, entry in result_cohort.cohort_results.get("region", {}).items():
    print(f"{region:<8} {entry['winner']:<12} {entry['confidence']:>12.3f} {entry['delta_accuracy']:>+12.3f}")

Global winner: baseline (confidence: 1.000)

Region   Winner         Confidence    Delta Acc
------------------------------------------------
EU       baseline            0.998       -0.050
US       baseline            0.996       -0.053
APAC     baseline            0.795       -0.027


## 8. stop_early — declare winner before min_samples

In [9]:
rng2 = np.random.default_rng(7)
n_small = 60   # well below min_samples=500
yt = rng2.integers(0, 2, n_small).tolist()
bp = [min(1, max(0, y + rng2.normal(0.05, 0.35))) for y in yt]
cp = [min(1, max(0, y + rng2.normal(0.40, 0.08))) for y in yt]  # challenger much better

# Without stop_early — stays inconclusive
exp_no = Experiment("no_early", min_samples=500, stop_early=False, min_confidence=0.7)
exp_no.log("baseline",   y_true=yt, y_pred=bp)
exp_no.log("challenger", y_true=yt, y_pred=cp)
r_no = exp_no.evaluate()
print(f"stop_early=False  →  winner={r_no.winner:<14} ready={r_no.ready}  conf={r_no.confidence:.3f}")

# With stop_early — declares winner early if confidence is high enough
exp_yes = Experiment("with_early", min_samples=500, stop_early=True, min_confidence=0.7)
exp_yes.log("baseline",   y_true=yt, y_pred=bp)
exp_yes.log("challenger", y_true=yt, y_pred=cp)
r_yes = exp_yes.evaluate()
print(f"stop_early=True   →  winner={r_yes.winner:<14} ready={r_yes.ready}  conf={r_yes.confidence:.3f}")

stop_early=False  →  winner=inconclusive   ready=False  conf=0.899
stop_early=True   →  winner=challenger     ready=True  conf=0.899


## 9. The three methods compared

In [10]:
yt = y_test.tolist()
bp = base_preds.tolist()
cp = chal_preds.tolist()

for method in ["bayesian", "sequential", "bandit"]:
    e = Experiment(f"method_{method}", method=method, min_samples=100, min_confidence=0.9)
    e.log("baseline",   y_true=yt, y_pred=bp)
    e.log("challenger", y_true=yt, y_pred=cp)
    r = e.evaluate()
    print(f"method={method:<12}  winner={r.winner:<14} conf={r.confidence:.4f}  ready={r.ready}")

method=bayesian      winner=challenger     conf=1.0000  ready=True
method=sequential    winner=challenger     conf=0.9600  ready=True
method=bandit        winner=challenger     conf=1.0000  ready=True


## 10. save() and load() — persisting the experiment

In [11]:
import json, os

save_path = "/tmp/churn_demo.evalbridge"

exp.save(save_path)

# Inspect what's in the file
with open(save_path) as f:
    payload = json.load(f)

print("Keys in the .evalbridge file:")
for k, v in payload.items():
    if k == "data":
        print(f"  {k}: baseline={len(v['baseline']['y_true'])} samples, challenger={len(v['challenger']['y_true'])} samples")
    else:
        print(f"  {k}: {v}")

Keys in the .evalbridge file:
  version: 0.1.0
  name: churn_v2_vs_v3
  method: bayesian
  min_confidence: 0.95
  min_samples: 100
  stop_early: False
  cohorts: {}
  alert_on_drift: False
  drift_threshold: 0.2
  created_at: 2026-06-02T10:32:19.644127+00:00
  saved_at: 2026-06-02T10:32:19.687380+00:00
  data: baseline=1200 samples, challenger=1200 samples
  cohort_data: {}


In [12]:
# Load it back and verify everything is preserved
exp_loaded = Experiment.load(save_path)

print(f"name          : {exp_loaded.name}")
print(f"method        : {exp_loaded.method}")
print(f"min_confidence: {exp_loaded.min_confidence}")
print(f"baseline n    : {len(exp_loaded._data['baseline']['y_true'])}")
print(f"challenger n  : {len(exp_loaded._data['challenger']['y_true'])}")

# Re-evaluate from loaded state
r_loaded = exp_loaded.evaluate()
print(f"\nRe-evaluated: winner={r_loaded.winner}, conf={r_loaded.confidence:.4f}")

os.remove(save_path)

name          : churn_v2_vs_v3
method        : bayesian
min_confidence: 0.95
baseline n    : 1200
challenger n  : 1200

Re-evaluated: winner=challenger, conf=1.0000


## 11. HTML report

In [13]:
# Generate the report without opening a browser
result.report(path="/tmp/experiment_report.html", open_browser=False)
print("Report written. Open /tmp/experiment_report.html in a browser to view it.")

[evalbridge] Report saved to /tmp/experiment_report.html
Report written. Open /tmp/experiment_report.html in a browser to view it.


## 12. Thread-safe logging (concurrent requests)

In [14]:
import threading

exp_thread = Experiment("thread_demo", min_samples=100)
n_threads = 8
logs_per_thread = 50

def worker(thread_id):
    rng_t = np.random.default_rng(thread_id)
    for _ in range(logs_per_thread):
        yt = rng_t.integers(0, 2, 5).tolist()
        bp = np.clip(rng_t.normal(0.5, 0.3, 5), 0, 1).tolist()
        cp = np.clip(rng_t.normal(0.6, 0.25, 5), 0, 1).tolist()
        exp_thread.log("baseline",   y_true=yt, y_pred=bp)
        exp_thread.log("challenger", y_true=yt, y_pred=cp)

threads = [threading.Thread(target=worker, args=(i,)) for i in range(n_threads)]
for t in threads: t.start()
for t in threads: t.join()

expected = n_threads * logs_per_thread * 5
actual_b = len(exp_thread._data['baseline']['y_true'])
actual_c = len(exp_thread._data['challenger']['y_true'])

print(f"Expected {expected} samples per model")
print(f"baseline:   {actual_b} — {'✓ no data lost' if actual_b == expected else '✗ data lost!'}")
print(f"challenger: {actual_c} — {'✓ no data lost' if actual_c == expected else '✗ data lost!'}")

Expected 2000 samples per model
baseline:   2000 — ✓ no data lost
challenger: 2000 — ✓ no data lost
